In [1]:
### 프로젝트 루트 경로 설정 및 YAML 설정 파일 로드
# 주피터 노트북이 `notebook/` 폴더 안에서 실행될 때 발생하는 경로 깨짐 문제를 방지하고, 프로젝트 전역 설정(`config/default.yaml`)을 불러옵니다.
import os
import sys
import yaml

# 주피터 노트북 실행 위치에 따른 프로젝트 루트 경로 자동 보정
current_dir = os.getcwd()
if current_dir.endswith("notebook"):
    os.chdir("..")

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 시스템 전역 설정 파일 로드
with open("config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("설정 로드 완료:", config["generation"])

설정 로드 완료: {'provider': 'openai', 'model': 'gpt-5-nano', 'temperature': 0.1, 'top_p': 0.95, 'max_tokens': 3000}


In [2]:
### 연동 테스트용 Mock 청크 데이터 로드
# 사전에 정의한 가상 제안요청서 청크 데이터(`sample_chunks.jsonl`)를 `SearchResult` 객체로 파싱합니다.
import json
from src.retriever import SearchResult

mock_results = []
file_path = "samples/processed/sample_chunks.jsonl"

# 가상 청크 JSONL 파일을 읽어 SearchResult 객체 리스트로 변환
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        result = SearchResult(
            chunk_id=data["chunk_id"],
            doc_id=data["doc_id"],
            text=data["text"],
            score=0.85,  # 가상 테스트용 임시 유사도 점수
            metadata=data["metadata"]
        )
        mock_results.append(result)

print(f"총 {len(mock_results)}개의 청크 데이터를 성공적으로 불러왔습니다!")

총 3개의 청크 데이터를 성공적으로 불러왔습니다!


In [3]:
### Test Case 1: 문서 내 다중 조건 복합 질문 테스트
# 여러 청크에 나뉘어 있는 정보(주민등록번호 제외 조건, 1년 보관 기간)를 하나의 답변으로 정확히 종합하는지 검증합니다.
from src.rag_engine import generate_answer
question_1 = "기존 시스템에서 사용자 정보 이전할 때 제외해야 하는 항목이 뭐야? 그리고 개인정보 다운로드 기록은 얼마나 보관해야 해?"
response_1 = generate_answer(question_1, mock_results, config)

print(f"Q: {question_1}\n")
print(f"A:\n{response_1['answer']}")

Q: 기존 시스템에서 사용자 정보 이전할 때 제외해야 하는 항목이 뭐야? 그리고 개인정보 다운로드 기록은 얼마나 보관해야 해?

A:
- 제외 대상 항목: 주민등록번호는 데이터 이전 대상에서 제외합니다.
- 개인정보 조회 및 다운로드 기록 보관 기간: 1년간 보관해야 합니다.

참고: sample_rfp.txt (2026년 공공 AI 학습지원 플랫폼 구축 사업)


In [4]:
### Test Case 2: 문서 외 질문에 대한 할루시네이션 방어 테스트
# 제공된 문서에 존재하지 않는 기능에 대해 아는 척 지어내지 않고, 단호하게 "확인할 수 없다"고 차단하는지 검증합니다.
question_2 = "이 플랫폼에 블록체인 연계 기능이 포함되어 있어?"
response_2 = generate_answer(question_2, mock_results, config)

print(f"Q: {question_2}\n")
print(f"A:\n{response_2['answer']}")

Q: 이 플랫폼에 블록체인 연계 기능이 포함되어 있어?

A:
- 제공된 문서(sample_rfp.txt)에서 블록체인 연계 기능의 포함 여부에 대한 언급은 확인되지 않습니다.
- 따라서 이 문서만으로는 블록체인 연계의 필요 여부를 판단할 수 없으며, 확인이 필요합니다.
- 블록체인 연계가 필요하다면 발주기관에 구체적 요구사항 확인 및 추가 명세 반영이 필요합니다.

참고: sample_rfp.txt


In [5]:
### Test Case 3: 제안 공고 조건 및 제출 방식 확인 테스트
# RFP의 마감일시와 제한 조건(이메일 제출 불가 등)을 정확하게 추출하여 안내하는지 검증합니다.
question_3 = "제안서 제출 마감일시는 언제이며, 이메일 제출이 가능한가요?"
response_3 = generate_answer(question_3, mock_results, config)

print(f"Q: {question_3}\n")
print(f"A:\n{response_3['answer']}")

Q: 제안서 제출 마감일시는 언제이며, 이메일 제출이 가능한가요?

A:
- 제안서 제출 마감일: 2026년 8월 14일 17시
- 이메일 제출 여부: 불가합니다. 이메일 제출은 인정되지 않습니다.
- 제출 방법: 나라장터를 통한 온라인 제출, 전자파일은 PDF 형식으로 등록합니다.

참고: sample_rfp.txt


In [6]:
### Test Case 4: 시스템 요구사항 및 제약 조건 추출 테스트
# 문서 내에 명시된 기능적 필수 조건(출처 표시, 근거 없을 시 응답 제한)을 누락 없이 요약하는지 검증합니다.
question_4 = "AI 학습 도우미 기능에서 답변을 제공할 때 지켜야 할 필수 조건은 뭐야?"
response_4 = generate_answer(question_4, mock_results, config)

print(f"Q: {question_4}\n")
print(f"A:\n{response_4['answer']}")

Q: AI 학습 도우미 기능에서 답변을 제공할 때 지켜야 할 필수 조건은 뭐야?

A:
- AI 학습 도우미의 답변은 사용된 교육자료의 제목과 위치를 함께 표시해야 합니다.
- 답변은 교육자료에 기반한 내용으로 한정되며, 교육자료에 없는 내용을 사실처럼 생성해서는 안 됩니다.
- 근거를 찾지 못하는 경우에는 "확인할 수 없습니다"라고 응답해야 합니다.

참고한 문서: sample_rfp.txt


In [7]:
### Test Case 5: 문서에 없는 부가 정책 질문에 대한 방어 테스트
# 예산이나 보안 외의 사내 복리후생 성격의 질문에 대해 추측하지 않고 근거 부재를 올바르게 안내하는지 검증합니다.
question_5 = "이 프로젝트에 투입되는 개발자들의 식대 지원 기준은 어떻게 돼?"
response_5 = generate_answer(question_5, mock_results, config)

print(f"Q: {question_5}\n")
print(f"A:\n{response_5['answer']}")

Q: 이 프로젝트에 투입되는 개발자들의 식대 지원 기준은 어떻게 돼?

A:
- 확인 결과: 본 RFP 문서(sample_rfp.txt)에는 개발자들의 식대 지원 기준에 대한 내용이 명시되어 있지 않아 확인할 수 없습니다.

- 필요 시: 관련 조항이나 정책이 담긴 추가 문서를 제공해 주시면 해당 내용을 바탕으로 자세히 분석해 드리겠습니다.

참고: sample_rfp.txt


In [8]:
from src.rag_engine import build_context, SYSTEM_RULE
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-5-nano", max_tokens=1000)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_RULE + "\n\nContext:\n{context}"),
    ("human", "{question}")
])
chain = prompt | llm

msg = chain.invoke({
    "context": build_context(mock_results),
    "question": "이 플랫폼에 블록체인 연계 기능이 포함되어 있어?"
})
print(msg.response_metadata)
print(repr(msg.content))

{'token_usage': {'completion_tokens': 767, 'prompt_tokens': 1437, 'total_tokens': 2204, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1280}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E1ls35QbY7iKqCd6YQN2tVWXIIXIy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}
'- 제공된 문서에서는 블록체인 연계 기능의 포함 여부를 확인할 수 없습니다. 문서의 주요 요구사항은 AI 학습 도우미, 학습 관리, 데이터 연계 방식(REST API), 보안 및 성능/가용성에 집중되어 있으며 블록체인에 대한 언급은 없습니다.\n- 따라서 블록체인 연계 기능이 포함되는지 여부를 확정하려면 추가 자료가 필요합니다.\n- 참고: sample_rfp.txt (2026년 공공 AI 학습지원 플랫폼 구축 사업)'
